<a href="https://colab.research.google.com/github/gustavopatinot/SECOP_Vec_Local/blob/main/notebooks_colab/corpus_secop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalacion librerías necesarias

In [ ]:
!pip install -q duckdb pyarrow requests

# MÓDULO 01: FÁBRICA DEL CORPUS Y BENCHMARK SECOP-VEC

In [ ]:
"""
MÓDULO 01: FÁBRICA DEL CORPUS Y BENCHMARK SECOP-VEC (ETAPA 1)
================================================================================
Proyecto de Tesis de Maestría:
    Búsqueda semántica a gran escala con memoria restringida:
    HNSW cuantizado aplicado a la contratación pública colombiana (SECOP II)
    en un computador de 8 GB de RAM.

Objetivo Específico 1:
    Construir y publicar un corpus limpio del SECOP II con sus embeddings,
    verdad de referencia y conjuntos de consultas.

Entorno de Ejecución:
    Google Colab Pro con persistencia transaccional en Google Drive.
================================================================================
"""

import os
import re
import json
import time
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import requests

# Configuración del registro de auditoría científica en español
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | [%(levelname)s] | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
registro_auditoria = logging.getLogger("FabricaCorpusSECOP")


class ConfiguracionFabricaSECOP:
    """Parámetros inmutables de gobernanza de datos para la Etapa 1."""

    # 1. Identificador oficial Socrata en datos.gov.co (SECOP II - Contratos Electrónicos)
    ID_RECURSO_SOCRATA: str = "jbjy-vk9h"
    URL_BASE_SOCRATA: str = f"https://www.datos.gov.co/resource/{ID_RECURSO_SOCRATA}.json"

    # 2. Parámetros de paginación y tolerancia a fallos de red
    TAMANO_LOTE_DESCARGA: int = 50_000
    MAXIMO_REINTENTOS_HTTP: int = 5
    TIEMPO_ESPERA_SEGUNDOS: int = 120

    # 3. Criterios metodológicos de curaduría del corpus (Capítulo 3 de la Tesis)
    ANIO_INICIO_CORTE: int = 2019
    ANIO_FIN_CORTE: int = 2025
    LONGITUD_MINIMA_CARACTERES: int = 30
    META_MINIMA_CORPUS: int = 1_000_000
    META_MAXIMA_CORPUS: int = 2_000_000
    CANTIDAD_CONSULTAS_EVALUACION: int = 10_000
    SEMILLA_DETERMINISTA: int = 42

    # 4. Rutas persistentes en Google Drive
    RUTA_RAIZ_DRIVE: Path = Path("/content/drive/MyDrive/SECOP_Vec_Tesis")
    RUTA_BRONCE: Path = RUTA_RAIZ_DRIVE / "01_bronce_socrata_parquet"
    RUTA_PLATA: Path = RUTA_RAIZ_DRIVE / "02_plata_corpus_deduplicado"
    RUTA_ORO: Path = RUTA_RAIZ_DRIVE / "03_oro_benchmark_secop_vec"
    RUTA_REPORTES_TESIS: Path = RUTA_RAIZ_DRIVE / "04_tablas_y_metricas_tesis"
    ARCHIVO_PUNTO_CONTROL: Path = RUTA_BRONCE / "estado_descarga_checkpoint.json"


def montar_almacenamiento_persistente_drive(config: ConfiguracionFabricaSECOP) -> None:
    """
    Monta Google Drive en el entorno de Google Colab Pro y garantiza la
    existencia de la arquitectura de directorios Medallón.
    """
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            registro_auditoria.info("Montando Google Drive para persistencia no volátil...")
            drive.mount("/content/drive")
    except ImportError:
        registro_auditoria.warning(
            "Ejecución fuera de Google Colab detectada. Usando sistema de archivos local."
        )

    for carpeta in [
        config.RUTA_BRONCE,
        config.RUTA_PLATA,
        config.RUTA_ORO,
        config.RUTA_REPORTES_TESIS,
    ]:
        carpeta.mkdir(parents=True, exist_ok=True)
    registro_auditoria.info(f"Estructura persistente verificada en: {config.RUTA_RAIZ_DRIVE}")


def obtener_token_socrata() -> Optional[str]:
    """
    Recupera de forma segura el token de Socrata desde los Secretos de Colab
    o variables de entorno sin exponer credenciales en el código fuente.
    """
    token = os.environ.get("SOCRATA_APP_TOKEN")
    if token:
        return token.strip()
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("SOCRATA_APP_TOKEN")
        if token:
            registro_auditoria.info("Token de Socrata cargado exitosamente desde Colab Secrets.")
            return token.strip()
    except Exception:
        pass

    registro_auditoria.warning(
        "No se encontró 'SOCRATA_APP_TOKEN'. La descarga funcionará pero con menor límite de tasa."
    )
    return None


class DescargadorTransaccionalSocrata:
    """
    Motor de ingesta desde la API SoQL de Socrata hacia archivos Parquet en Google Drive
    con puntos de control (checkpoints) atómicos para reanudación automática.
    """

    def __init__(self, config: ConfiguracionFabricaSECOP, token_app: Optional[str]):
        self.config = config
        self.sesion = requests.Session()
        self.sesion.headers.update({"Accept": "application/json"})
        if token_app:
            self.sesion.headers.update({"X-App-Token": token_app})

    def _leer_punto_control(self) -> Dict[str, int]:
        if self.config.ARCHIVO_PUNTO_CONTROL.exists():
            with open(self.config.ARCHIVO_PUNTO_CONTROL, "r", encoding="utf-8") as f:
                return json.load(f)
        return {"ultimo_offset_completado": 0, "lotes_guardados": 0}

    def _guardar_punto_control(self, offset: int, lotes: int) -> None:
        estado = {
            "ultimo_offset_completado": offset,
            "lotes_guardados": lotes,
            "timestamp_actualizacion": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        archivo_temporal = self.config.ARCHIVO_PUNTO_CONTROL.with_suffix(".tmp")
        with open(archivo_temporal, "w", encoding="utf-8") as f:
            json.dump(estado, f, indent=2, ensure_ascii=False)
        archivo_temporal.replace(self.config.ARCHIVO_PUNTO_CONTROL)

    def ejecutar_ingesta_incremental(
        self,
        descargar_95_columnas_completas: bool = False,
        limite_maximo_registros: Optional[int] = None,
    ) -> None:
        """
        Descarga los registros desde Socrata en bloques Parquet comprimidos con Zstandard.
        Si `descargar_95_columnas_completas=False`, proyecta desde SoQL las columnas
        requeridas por la metodología experimental para acelerar la transferencia 6x.
        """
        estado = self._leer_punto_control()
        offset_actual = estado["ultimo_offset_completado"]
        indice_lote = estado["lotes_guardados"]

        columnas_nucleo_tesis = (
            "id_contrato, proceso_de_compra, objeto_del_contrato, descripcion_del_proceso, "
            "codigo_de_categoria_principal, departamento, ciudad, nombre_entidad, nit_entidad, "
            "modalidad_de_contratacion, tipo_de_contrato, valor_del_contrato, fecha_de_firma, "
            "estado_contrato"
        )

        registro_auditoria.info(
            f"Iniciando ingesta Socrata desde offset={offset_actual:,} (Lote #{indice_lote})..."
        )

        while True:
            if limite_maximo_registros and offset_actual >= limite_maximo_registros:
                registro_auditoria.info("Límite máximo de registros de prueba alcanzado.")
                break

            parametros_soql = {
                "$limit": self.config.TAMANO_LOTE_DESCARGA,
                "$offset": offset_actual,
                "$order": ":id ASC",
            }
            if not descargar_95_columnas_completas:
                parametros_soql["$select"] = columnas_nucleo_tesis

            datos_lote = self._solicitar_lote_con_reintento(parametros_soql)
            if not datos_lote:
                registro_auditoria.info("Descarga masiva completada: fin del conjunto de datos.")
                break

            # Normalización preventiva de tipos como texto/cadena para evitar conflictos de esquema
            registros_normalizados = [
                { str(k): (str(v) if v is not None else None) for k, v in fila.items() }
                for fila in datos_lote
            ]

            tabla_arrow = pa.Table.from_pylist(registros_normalizados)
            nombre_archivo_lote = (
                self.config.RUTA_BRONCE / f"secop_bronce_lote_{indice_lote:05d}.parquet"
            )
            pq.write_table(tabla_arrow, nombre_archivo_lote, compression="zstd")

            offset_actual += len(datos_lote)
            indice_lote += 1
            self._guardar_punto_control(offset_actual, indice_lote)

            registro_auditoria.info(
                f"Lote #{indice_lote:04d} persistido en Drive | "
                f"Acumulado: {offset_actual:,} contratos."
            )

            if len(datos_lote) < self.config.TAMANO_LOTE_DESCARGA:
                registro_auditoria.info("Último bloque parcial recibido. Ingesta finalizada.")
                break

    def _solicitar_lote_con_reintento(self, parametros: Dict) -> List[Dict]:
        for intento in range(1, self.config.MAXIMO_REINTENTOS_HTTP + 1):
            try:
                respuesta = self.sesion.get(
                    self.config.URL_BASE_SOCRATA,
                    params=parametros,
                    timeout=self.config.TIEMPO_ESPERA_SEGUNDOS,
                )
                respuesta.raise_for_status()
                return respuesta.json()
            except Exception as error_red:
                espera = 2 ** intento
                registro_auditoria.warning(
                    f"Intento {intento}/{self.config.MAXIMO_REINTENTOS_HTTP} falló ({error_red}). "
                    f"Reintentando en {espera}s..."
                )
                time.sleep(espera)
        raise RuntimeError("Fallo crítico de red tras agotar los reintentos contra Socrata API.")


class CuradorCientificoSECOPVec:
    """
    Ejecuta fuera de núcleo (out-of-core con DuckDB) la limpieza, deduplicación,
    descomposición jerárquica del código UNSPSC y partición en Corpus y Consultas.
    """

    def __init__(self, config: ConfiguracionFabricaSECOP):
        self.config = config
        self.conexion = duckdb.connect(database=":memory:")
        self.conexion.execute("PRAGMA threads=4;")

    def construir_corpus_y_consultas(self, patron_parquet_origen: Optional[str] = None) -> Dict:
        """
        Procesa los archivos Parquet de la capa Bronce (o los que ya tengas en Drive),
        genera el corpus deduplicado de 1 a 2 millones de documentos únicos y
        extrae las 10.000 consultas estratificadas para la verdad de referencia.
        """
        ruta_lectura = patron_parquet_origen or str(self.config.RUTA_BRONCE / "*.parquet")
        registro_auditoria.info(f"Auditando archivos Parquet desde: {ruta_lectura}")

        # 1. Conteo bruto para tabla del Capítulo 3
        total_bruto = self.conexion.execute(
            f"SELECT COUNT(*) FROM read_parquet('{ruta_lectura}', union_by_name=true)"
        ).fetchone()[0]

        archivo_plata = self.config.RUTA_PLATA / "secop_plata_deduplicado.parquet"

        # 2. Consulta SQL de curaduría, normalización textual, extracción UNSPSC y deduplicación
        consulta_curaduria = f"""
        COPY (
            WITH proyeccion_limpia AS (
                SELECT
                    CAST(id_contrato AS VARCHAR) AS id_contrato,
                    CAST(proceso_de_compra AS VARCHAR) AS numero_proceso,
                    TRIM(REGEXP_REPLACE(
                        COALESCE(NULLIF(TRIM(objeto_del_contrato), ''), TRIM(descripcion_del_proceso)),
                        '\\s+', ' ', 'g'
                    )) AS descripcion_objeto,
                    CAST(codigo_de_categoria_principal AS VARCHAR) AS codigo_unspsc_original,
                    REGEXP_EXTRACT(CAST(codigo_de_categoria_principal AS VARCHAR), '([0-9]{{8}})', 1) AS unspsc_8digitos,
                    UPPER(TRIM(CAST(departamento AS VARCHAR))) AS departamento,
                    UPPER(TRIM(CAST(nombre_entidad AS VARCHAR))) AS entidad,
                    UPPER(TRIM(CAST(modalidad_de_contratacion AS VARCHAR))) AS modalidad,
                    UPPER(TRIM(CAST(tipo_de_contrato AS VARCHAR))) AS tipo_contrato,
                    TRY_CAST(REGEXP_REPLACE(CAST(valor_del_contrato AS VARCHAR), '[^0-9.]', '', 'g') AS DOUBLE) AS valor_contrato,
                    TRY_CAST(SUBSTRING(CAST(fecha_de_firma AS VARCHAR), 1, 4) AS INTEGER) AS anio_firma
                FROM read_parquet('{ruta_lectura}', union_by_name=true)
            ),
            filtrado_calidad AS (
                SELECT
                    *,
                    SUBSTRING(unspsc_8digitos, 1, 2) AS unspsc_segmento,
                    SUBSTRING(unspsc_8digitos, 1, 4) AS unspsc_familia,
                    SUBSTRING(unspsc_8digitos, 1, 6) AS unspsc_clase,
                    MD5(LOWER(descripcion_objeto)) AS hash_descripcion,
                    CASE
                        WHEN tipo_contrato LIKE '%PRESTACI%SERVICIOS%'
                          OR modalidad LIKE '%PRESTACI%SERVICIOS%'
                        THEN TRUE ELSE FALSE
                    END AS es_prestacion_servicios
                FROM proyeccion_limpia
                WHERE descripcion_objeto IS NOT NULL
                  AND LENGTH(descripcion_objeto) >= {self.config.LONGITUD_MINIMA_CARACTERES}
                  AND LENGTH(unspsc_8digitos) = 8
                  AND anio_firma BETWEEN {self.config.ANIO_INICIO_CORTE} AND {self.config.ANIO_FIN_CORTE}
                  AND valor_contrato IS NOT NULL
                  AND valor_contrato > 0
            ),
            deduplicado_canonico AS (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY hash_descripcion
                        ORDER BY anio_firma DESC, id_contrato ASC
                    ) AS rango_unicidad,
                    COUNT(*) OVER (PARTITION BY hash_descripcion) AS frecuencia_repeticion
                FROM filtrado_calidad
            )
            SELECT
                id_contrato,
                numero_proceso,
                descripcion_objeto,
                unspsc_8digitos,
                unspsc_segmento,
                unspsc_familia,
                unspsc_clase,
                departamento,
                entidad,
                modalidad,
                tipo_contrato,
                es_prestacion_servicios,
                valor_contrato,
                anio_firma,
                frecuencia_repeticion
            FROM deduplicado_canonico
            WHERE rango_unicidad = 1
            LIMIT {self.config.META_MAXIMA_CORPUS}
        ) TO '{archivo_plata}' (FORMAT PARQUET, COMPRESSION ZSTD);
        """
        self.conexion.execute(consulta_curaduria)

        # 3. Separación en Conjunto de Consultas (10.000 consultas con clase UNSPSC con >= 10 contratos)
        #    y Corpus de Indexación (SECOP-Vec v1)
        archivo_consultas_oro = self.config.RUTA_ORO / "secop_vec_consultas_10k.parquet"
        archivo_corpus_oro = self.config.RUTA_ORO / "secop_vec_corpus_base.parquet"

        self.conexion.execute(f"""
        COPY (
            WITH clases_densas AS (
                SELECT unspsc_clase
                FROM read_parquet('{archivo_plata}')
                GROUP BY unspsc_clase
                HAVING COUNT(*) >= 10
            ),
            candidatos_consulta AS (
                SELECT p.*
                FROM read_parquet('{archivo_plata}') p
                INNER JOIN clases_densas c ON p.unspsc_clase = c.unspsc_clase
                USING SAMPLE {self.config.CANTIDAD_CONSULTAS_EVALUACION} ROWS (reservoir, {self.config.SEMILLA_DETERMINISTA})
            )
            SELECT * FROM candidatos_consulta
        ) TO '{archivo_consultas_oro}' (FORMAT PARQUET, COMPRESSION ZSTD);
        """)

        self.conexion.execute(f"""
        COPY (
            SELECT p.*
            FROM read_parquet('{archivo_plata}') p
            LEFT JOIN read_parquet('{archivo_consultas_oro}') q
              ON p.id_contrato = q.id_contrato
            WHERE q.id_contrato IS NULL
        ) TO '{archivo_corpus_oro}' (FORMAT PARQUET, COMPRESSION ZSTD);
        """)

        # 4. Cálculo de métricas descriptivas para el Capítulo 3 de la Tesis
        metricas = self.conexion.execute(f"""
            SELECT
                COUNT(*) AS total_corpus_unico,
                SUM(CASE WHEN es_prestacion_servicios THEN 1 ELSE 0 END) AS total_prestacion_servicios,
                COUNT(DISTINCT unspsc_segmento) AS segmentos_unicos,
                COUNT(DISTINCT unspsc_familia) AS familias_unicas,
                COUNT(DISTINCT unspsc_clase) AS clases_unicas,
                ROUND(AVG(LENGTH(descripcion_objeto)), 2) AS longitud_media_caracteres,
                 QUANTILE_CONT(LENGTH(descripcion_objeto), 0.5) AS longitud_mediana_caracteres
            FROM read_parquet('{archivo_corpus_oro}')
        """).fetchone()

        total_consultas = self.conexion.execute(
            f"SELECT COUNT(*) FROM read_parquet('{archivo_consultas_oro}')"
        ).fetchone()[0]

        resumen_capitulo_3 = {
            "registros_brutos_auditados": int(total_bruto),
            "descripciones_unicas_corpus_oro": int(metricas[0]),
            "consultas_verdad_referencia": int(total_consultas),
            "contratos_prestacion_servicios_en_corpus": int(metricas[1]),
            "segmentos_unspsc_cubiertos": int(metricas[2]),
            "familias_unspsc_cubiertas": int(metricas[3]),
            "clases_unspsc_cubiertas": int(metricas[4]),
            "longitud_media_caracteres": float(metricas[5]),
            "longitud_mediana_caracteres": float(metricas[6]),
        }

        ruta_json_tesis = self.config.RUTA_REPORTES_TESIS / "tabla_3_1_estadisticas_corpus.json"
        with open(ruta_json_tesis, "w", encoding="utf-8") as f:
            json.dump(resumen_capitulo_3, f, indent=2, ensure_ascii=False)

        registro_auditoria.info(f"Estadísticas del Capítulo 3 exportadas en: {ruta_json_tesis}")
        return resumen_capitulo_3

# Ejecución primer módulo

In [ ]:
config = ConfiguracionFabricaSECOP()
montar_almacenamiento_persistente_drive(config)
token = obtener_token_socrata()

descargador = DescargadorTransaccionalSocrata(config, token)
# descargar_95_columnas_completas=True si deseas guardar las 95 columnas en Bronce
descargador.ejecutar_ingesta_incremental(descargar_95_columnas_completas=True)

curador = CuradorCientificoSECOPVec(config)
estadisticas = curador.construir_corpus_y_consultas()
print(json.dumps(estadisticas, indent=2, ensure_ascii=False))

Mounted at /content/drive


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "registros_brutos_auditados": 5714490,
  "descripciones_unicas_corpus_oro": 1990000,
  "consultas_verdad_referencia": 10000,
  "contratos_prestacion_servicios_en_corpus": 1621997,
  "segmentos_unspsc_cubiertos": 56,
  "familias_unspsc_cubiertas": 405,
  "clases_unspsc_cubiertas": 2299,
  "longitud_media_caracteres": 251.51,
  "longitud_mediana_caracteres": 234.0
}
